In [ ]:
using JLD2
using LinearAlgebra
using Unitful

using Plots

Plots.default(
    #fontfamily = "JuliaMono-Regular",
    #titlefont = (10, "JuliaMono-Regular"),
    #legendfont = (8, "JuliaMono-Regular"),
    #guidefont = (8, "JuliaMono-Regular", :black),
    #tickfont = (8, "JuliaMono-Regular", :black),
    rightmargin=5Plots.mm,
    leftmargin=5Plots.mm,
    bottommargin=5Plots.mm,
    format=:svg
)

using RetrievalToolbox; const RE = RetrievalToolbox;

In [ ]:
ENV["XRTM_PROGRESS"] = 1

# (1) Basic exercise

In [ ]:
wavelength_start = 4110.0u"nm"
wavelength_end = 4115.0u"nm"
wavelength_ref = (wavelength_start + wavelength_end) / 2

gas_name = "CO2"

In [ ]:
# Alternative:
# (but beware - no surface temp sensitivity here!!)
#=
wavelength_start = 4200.0u"nm"
wavelength_end = 4205.0u"nm"
wavelength_ref = (wavelength_start + wavelength_end) / 2

gas_name = "CO2"
=#

In [ ]:
# Full window
#=
wavelength_start = 4100.0u"nm"
wavelength_end = 4200.0u"nm"
wavelength_ref = (wavelength_start + wavelength_end) / 2

gas_name = "CO2"
=#

In [ ]:
# Create a 10-level (9-layer) Earth atmosphere based on some
# model data shipped with RetrievalToolbox
my_source_atmosphere = RE.create_example_atmosphere("US-midwest-summer", 10);

In [ ]:
# Fetch spectroscopy

if isfile("demo_spectroscopy_thermal.jld2")

    @info "Loading spectroscopy from local file.."
    my_spectroscopy = jldopen("demo_spectroscopy_thermal.jld2")["my_spectroscopy"]

else
    # Request a BIG spectroscopy file, just in case we want to do
    # different things later on..
    my_spectroscopy = RE.create_ABSCO_from_HITRAN(
        gas_name,
        4100.0u"nm", # put some buffer around our requested wavelengths
        4400.0u"nm";
        wavelength_output=true,
        h2o_broadening=false
    )

    jldsave("demo_spectroscopy_thermal.jld2"; my_spectroscopy)
end


In [ ]:
my_gas = RE.GasAbsorber(
    gas_name,
    my_spectroscopy,
    fill(400.0, 10), # CO2 is ~400ppm
    Unitful.ppm # Unit of VMR profile
)

In [ ]:
my_spectral_window = RE.spectralwindow_from_ABSCO(
    "$(gas_name)_window",
    wavelength_start |> ustrip,
    wavelength_end |> ustrip,
    wavelength_ref |> ustrip,
    1.0, # Buffer length in same units as everything else
    my_spectroscopy,
    unit(wavelength_start)
)

In [ ]:
dλ = 0.01u"nm"
Nsamples = (wavelength_end - wavelength_start) / dλ |> Int

In [ ]:
my_dispersion = RE.SimplePolynomialDispersion(
    [wavelength_start, dλ],
    0:Nsamples - 1,
    my_spectral_window
)

In [ ]:
my_gas_scale_sve = RE.GasLevelScalingFactorSVE(
    1,
    10,
    my_gas,
    Unitful.NoUnits,
    1.0, # First guess
    1.0, # Prior value
    1.0 # Prior covariance
)

In [ ]:
my_T_offset_sve = RE.TemperatureOffsetSVE(
    u"K",
    0.0,
    0.0,
    10.0
)

In [ ]:
my_surface_T_sve = RE.SurfaceTemperatureSVE(
    u"K",
    300.0,
    300.0,
    15.0
)

In [ ]:
my_state_vector = RE.RetrievalStateVector([my_gas_scale_sve,  my_T_offset_sve, my_surface_T_sve]);
my_state_vector.state_vector_elements

In [ ]:
# No need for a solar model, since we do thermal
my_solar_model = RE.NoSolarModel()

In [ ]:
elements = [
    my_gas,
    RE.ThermalAtmosphereIsotropic(),
    RE.ThermalSurfaceIsotropic(300.0, u"K")
    ]

In [ ]:
# Push into the atmosphere object, and empty out beforehand
empty!(my_source_atmosphere.atm_elements)
for el in elements
    push!(my_source_atmosphere.atm_elements, el)
end

In [ ]:
N1 = 2 * length(my_spectroscopy.wavelength)
N2 = 2 * length(my_dispersion.detector_samples)

my_instrument_buffer = RE.InstrumentBuffer(
    zeros(Float64, N1),
    zeros(Float64, N1),
    zeros(Float64, N2),
);

my_rt_buffer = RE.ScalarRTBuffer(
    Dict(my_spectral_window => my_dispersion), # Already a SpectralWindows -> Dispersion dictionary
    RE.ScalarRadiance(Float64, N2), # Hold the radiance - we use ScalarRadiance because we don't need polarization
    Dict(sve => RE.ScalarRadiance(Float64, N2) for sve in my_state_vector.state_vector_elements),
    Dict(my_spectral_window => zeros(Int, 0)), # Hold the detector indices
    u"W/m^2/sr/µm" # Radiance units for the forward model, these should match the L1 observation units
);

my_buffer = RE.EarthAtmosphereBuffer(
    my_state_vector,
    my_spectral_window,
    [(:Lambert, 1)],
    my_source_atmosphere,
    Dict(my_spectral_window => my_solar_model),
    [:XRTM],
    RE.ScalarRadiance, # Use ScalarRadiance for high-res radiance calculations
    my_rt_buffer,
    my_instrument_buffer,
    10, # The number of retrieval or RT pressure levels
    my_source_atmosphere.N_met_level, # The number of meteorological pressure levels, as given by the atmospheric inputs
    Float64 # The chosen Float data type (e.g. Float16, Float32, Float64)    
);

In [ ]:
#=

    We are doing 2 passes through the RT model:

    1) single-scattering with a solar source, no thermal emission (solver: single)
    2) multiple-scattering with thermal sources, with a solar source (solver: eig_bvp)

    We configure our RT system such that contributions are ADDED TOGETHER.

    This is needed because we compile XRTM with #define DO_NOT_ADD_SFI_SS 1,
    and therefore single scatter contributions are NOT added in a call to the 
    "eig_bvp" multiple-scatter solver. Also, note that thermal emission can only
    be computed with certain solvers, and "single" is not one of them. Following
    solvers DO NOT support thermal sources:
        
        four_stream
        mem_bvp
        single
        six_stream
        sos
        two_os
        two_stream

    However THESE DO:

        doub_add
        eig_add
        eig_bvp
        pade_add
        
    (as of March 2026)

    Alternatively, users can compile XRTM with #define DO_NOT_ADD_SFI_SS 0, in which
    case XRTM internally computes the single-scatter contributions when "eig_bvp" is
    used, and 

=#

empty!(my_buffer.rt[my_spectral_window].model_options)

options_ss = Dict()
#=
options_ss["solvers"] = ["single"]
options_ss["streams"] = 2
options_ss["add"] = true # ADDS the RT results to the result container
options_ss["options"] = [
    "output_at_levels",
    "source_solar", # Use a solar source
    "psa",
    "sfi",
    "calc_derivs"
]
push!(my_buffer.rt[my_spectral_window].model_options, options_ss)
=#

options_ms = Dict()
options_ms["solvers"] = ["eig_bvp"]
options_ms["streams"] = 2
options_ms["add"] = false
options_ms["options"] = [
    "output_at_levels",
    "source_thermal", # Use a thermal source
    "source_solar", # Use a solar source
    "psa",
    "sfi",
    "calc_derivs"
]
push!(my_buffer.rt[my_spectral_window].model_options, options_ms)

In [ ]:
# Produce the correct indices (mappings between dispersion and buffer positions)
RE.calculate_indices!(my_buffer)

In [ ]:
# Set our custom pressure grid at which the gas VMRs are defined
my_buffer.scene.atmosphere.pressure_levels[:] = [
    1.0, # 1
    1_00.0, # 2
    10_00.0, # 3
    30_00.0, # 4
    100_00.0, # 5
    200_00.0, # 6
    400_00.0, # 7
    700_00.0, # 8
    800_00.0, # 9
   1000_00.0, # 10 - this is the surface pressure!
];

In [ ]:
my_buffer.scene.location

In [ ]:
# Calculate the mid-layer altitudes and gravity values
# (also re-calculates ALL relevant mid-layer values from level quantities)
RE.calculate_altitude_and_gravity!(my_buffer.scene)

In [ ]:
# Set the surface reflectance to some value
my_buffer.scene.surfaces[my_spectral_window].kernels[1].coefficients[1] = 0.25

In [ ]:
# Let RetrievalToolbox calculate the optical properties due to gas absorption
RE.calculate_earth_optical_properties!(
    my_buffer.rt[my_spectral_window].optical_properties,
    my_buffer.scene,
    my_state_vector
)

In [ ]:
# Calculate the solar irradiance
# (for a NoSolarModel, this will be zeros)
RE.calculate_solar_irradiance!(
    my_buffer.rt[my_spectral_window],
    my_spectral_window,
    my_buffer.rt[my_spectral_window].solar_model
);

In [ ]:
# Clear out RT results
RE.clear!(my_buffer.rt[my_spectral_window])

# Calculate the TOA radiance with XRTM
@time RE.calculate_radiances_and_jacobians!(my_buffer.rt[my_spectral_window])

In [ ]:
p = plot(
    my_spectral_window.wavelength_grid * my_spectral_window.wavelength_unit,
    my_buffer.rt[my_spectral_window].hires_radiance,
    label="High resolution model",
    ylabel="Radiance [$(my_buffer.rt[my_spectral_window].radiance_unit)]"
)
display(p)

In [ ]:
my_isrf = RE.GaussISRF(0.1, u"nm") # Specify FWHM here

In [ ]:
@time RE.apply_isrf_to_spectrum!(
    my_instrument_buffer,
    my_isrf,
    my_dispersion,
    my_buffer.rt[my_spectral_window].hires_radiance.I
)

In [ ]:
p = plot(
    my_spectral_window.wavelength_grid * my_spectral_window.wavelength_unit,
    my_buffer.rt[my_spectral_window].hires_radiance,
    label="High resolution model",
    linewidth=0.5, ylabel="Radiance [1]"
)

plot!(p,
    my_dispersion.wavelength,
    my_instrument_buffer.low_res_output[my_rt_buffer.indices[my_spectral_window]],
    label="Instrument-level",
    linewidth=2.0,
    marker=:circle, markersize=2
)
display(p)

In [ ]:
plot(
    my_spectral_window.ww_grid * my_spectral_window.ww_unit,
    my_buffer.rt[my_spectral_window].hires_jacobians[my_gas_scale_sve],
    label="∂I/∂c (c = gas scale factor)"
)

In [ ]:
plot(
    my_spectral_window.ww_grid * my_spectral_window.ww_unit,
    my_buffer.rt[my_spectral_window].hires_jacobians[my_T_offset_sve],
    label="∂I/∂ΔT"
)

In [ ]:
plot(
    my_spectral_window.ww_grid * my_spectral_window.ww_unit,
    my_buffer.rt[my_spectral_window].hires_jacobians[my_surface_T_sve],
    label="∂I/∂Tsurf"
)

In [ ]:
my_buffer.rt[my_spectral_window].wfunctions_map

In [ ]:
p = plot()
for (l,i) in enumerate(my_buffer.rt[my_spectral_window].wfunctions_map["dI_dblevel"])
    plot!(p,
        my_spectral_window.wavelength_grid,
        my_buffer.rt[my_spectral_window].hires_wfunctions[i],
        label="∂I/∂b$(l)"
    )
end
display(p)

In [ ]:
plot(
    my_buffer.scene.atmosphere.temperature_levels * my_buffer.scene.atmosphere.temperature_unit,
    my_buffer.scene.atmosphere.met_pressure_levels * my_buffer.scene.atmosphere.met_pressure_unit,
    linewidth=3,
    yscale=:log10,
    label=nothing
    );

hline!(
    my_buffer.scene.atmosphere.pressure_levels * my_buffer.scene.atmosphere.pressure_unit,
    linestyle=:dash, linecolor=:gray, label="Pressure grid"
)
    
yflip!(true)

# (2) Using solvers, inspect error analysis

In [ ]:
my_fake_forward_model(sv::RetrievalStateVector; kwargs) = true;

In [ ]:
my_state_vector

In [ ]:
my_fake_measurement = zeros(length(my_dispersion.detector_samples));
my_fake_noise = ones(length(my_dispersion.detector_samples)) * 0.0003;

In [ ]:
Sa = Diagonal([sv.prior_covariance for sv in my_state_vector.state_vector_elements])

In [ ]:
my_solver = RE.IMAPSolver(
    my_fake_forward_model,
    my_state_vector,
    Sa,
    99,
    10.0,
    Dict(my_spectral_window => my_dispersion),
    my_rt_buffer.indices,
    my_rt_buffer.radiance,
    my_rt_buffer.jacobians,
    Dict(my_dispersion => my_fake_measurement),
    Dict(my_dispersion => my_fake_noise),
);

In [ ]:
RE.calculate_indices!(my_buffer)

In [ ]:
RE.apply_isrf_to_spectrum!(
    my_instrument_buffer,
    my_isrf,
    my_dispersion,
    my_buffer.rt[my_spectral_window].hires_radiance.I
)

# This may need a nice convenience function!
my_rt_buffer.radiance.I[my_rt_buffer.indices[my_spectral_window]] =
    my_instrument_buffer.low_res_output[my_dispersion.index] *
    my_buffer.rt[my_spectral_window].radiance_unit / my_rt_buffer.radiance_unit;

In [ ]:
plot(
    RE.get_wavelength(my_solver),
    RE.get_modeled(my_solver),
    label="noiseless", linewidth=3
)

# add noise
noisified = RE.get_modeled(my_solver) .+ (randn(length(my_dispersion.index)) .* my_fake_noise)

plot!(
    RE.get_wavelength(my_solver),
    noisified,
    label="noisy"
    )

In [ ]:
# We have to do the ISRF application for all Jacobians too:
@time for sve in my_state_vector.state_vector_elements
    
    RE.apply_isrf_to_spectrum!(
        my_instrument_buffer,
        my_isrf,
        my_dispersion,
        my_buffer.rt[my_spectral_window].hires_jacobians[sve].I
    )

    # Move over to RT Buffer object
    my_rt_buffer.jacobians[sve].I[my_rt_buffer.indices[my_spectral_window]] =
        my_instrument_buffer.low_res_output[my_dispersion.index]

end

$\mathbf{K} = \partial\mathrm{F}(\mathbf{x}) / \partial \mathbf{x}$

In [ ]:
# Create the Jacobian matrix
K = RE.create_K_from_solver(my_solver);

In [ ]:
size(K)

In [ ]:
p = plot()
for (idx, sve) in enumerate(my_state_vector.state_vector_elements)
    plot!(my_dispersion.wavelength, K[:,idx], label="$(sve)")
end
display(p)

In [ ]:
q = RE.calculate_OE_quantities(my_solver);

In [ ]:
q.Shat

In [ ]:
RE.print_posterior(my_solver)

In [ ]:
C = similar(q.Shat)

for i in axes(C, 1), j in axes(C, 2)
    C[i,j] = q.Shat[i,j] / sqrt(q.Shat[i,i] * q.Shat[j,j])
end

C

In [ ]:
my_buffer.scene.surfaces[my_spectral_window].kernels[1].coefficients